In [11]:
import pandas as pd
import numpy as np

# Load raw data
df = pd.read_csv("../data/raw/deliveries_raw.csv")
print(f"Loaded raw data: {df.shape[0]} rows, {df.shape[1]} columns")

# Make a copy we'll clean — never modify the raw load
df_clean = df.copy()

Loaded raw data: 45593 rows, 20 columns


In [12]:

# ISSUE 1 FIX: Convert hidden "NaN " strings to real np.nan

# These columns have missing values stored as the literal string "NaN "
# Pandas can't detect them until we explicitly replace them.

# Strip all whitespace from text columns FIRST (fixes Issue 3 too)
text_cols = df_clean.select_dtypes(include='object').columns
for col in text_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip()

# Now replace "NaN" strings with actual NaN
df_clean = df_clean.replace("NaN", np.nan)

# Verify: nulls should now be detectable
print("Real nulls after conversion:")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])

Real nulls after conversion:
Delivery_person_Age        1854
Delivery_person_Ratings    1908
Time_Orderd                1731
Road_traffic_density        601
multiple_deliveries         993
Festival                    228
City                       1200
dtype: int64


In [13]:
# ISSUE 2 FIX: Remove garbage text prefixes

# "conditions Sunny" -> "Sunny", "(min) 24" -> "24"

df_clean['Weatherconditions'] = df_clean['Weatherconditions'].str.replace(
    'conditions ', '', regex=False
)

df_clean['Time_taken(min)'] = df_clean['Time_taken(min)'].str.replace(
    '(min) ', '', regex=False
)

# Verify
print("Weather samples:", df_clean['Weatherconditions'].dropna().head(3).tolist())
print("Time_taken samples:", df_clean['Time_taken(min)'].dropna().head(3).tolist())

Weather samples: ['Sunny', 'Stormy', 'Sandstorms']
Time_taken samples: ['24', '33', '26']


In [14]:
# ISSUE 7 FIX: Rename columns to clean snake_case

# SQL-friendly, consistent, no typos

df_clean = df_clean.rename(columns={
    'ID': 'order_id',
    'Delivery_person_ID': 'courier_id',
    'Delivery_person_Age': 'courier_age',
    'Delivery_person_Ratings': 'courier_rating',
    'Restaurant_latitude': 'restaurant_lat',
    'Restaurant_longitude': 'restaurant_lng',
    'Delivery_location_latitude': 'delivery_lat',
    'Delivery_location_longitude': 'delivery_lng',
    'Order_Date': 'order_date',
    'Time_Orderd': 'time_ordered',
    'Time_Order_picked': 'time_picked',
    'Weatherconditions': 'weather',
    'Road_traffic_density': 'traffic',
    'Vehicle_condition': 'vehicle_condition',
    'Type_of_order': 'order_type',
    'Type_of_vehicle': 'vehicle_type',
    'multiple_deliveries': 'multi_deliveries',
    'Festival': 'is_festival',
    'City': 'city_type',
    'Time_taken(min)': 'delivery_minutes'
})

print("New columns:", df_clean.columns.tolist())

New columns: ['order_id', 'courier_id', 'courier_age', 'courier_rating', 'restaurant_lat', 'restaurant_lng', 'delivery_lat', 'delivery_lng', 'order_date', 'time_ordered', 'time_picked', 'weather', 'traffic', 'vehicle_condition', 'order_type', 'vehicle_type', 'multi_deliveries', 'is_festival', 'city_type', 'delivery_minutes']


In [15]:
# ISSUE 4 FIX: Correct "Metropolitian" typo


df_clean['city_type'] = df_clean['city_type'].replace('Metropolitian', 'Metropolitan')

# Verify
print(df_clean['city_type'].value_counts(dropna=False))

city_type
Metropolitan    34093
Urban           10136
NaN              1200
Semi-Urban        164
Name: count, dtype: int64


In [16]:
# ==============================================================
# DROP ROWS — per decisions in data_quality_issues.md
# ==============================================================

rows_before = len(df_clean)
print(f"Rows before drops: {rows_before}")

# Drop rows with null time_ordered (can't impute timestamps)
df_clean = df_clean.dropna(subset=['time_ordered'])
print(f"After dropping null time_ordered: {len(df_clean)} rows (-{rows_before - len(df_clean)})")

rows_before = len(df_clean)

# Drop rows with null city_type (critical for geographic analysis)
df_clean = df_clean.dropna(subset=['city_type'])
print(f"After dropping null city_type: {len(df_clean)} rows (-{rows_before - len(df_clean)})")

rows_before = len(df_clean)

# Drop rows with invalid restaurant coordinates
df_clean = df_clean[df_clean['restaurant_lat'] != 0]
print(f"After dropping zero coords: {len(df_clean)} rows (-{rows_before - len(df_clean)})")

rows_before = len(df_clean)

# Drop rows where latitude is below India's southern tip (6°)
df_clean = df_clean[df_clean['restaurant_lat'] >= 6]
print(f"After dropping lat < 6: {len(df_clean)} rows (-{rows_before - len(df_clean)})")

print(f"\nFinal row count: {len(df_clean)}")

Rows before drops: 45593
After dropping null time_ordered: 43862 rows (-1731)


After dropping null city_type: 42718 rows (-1144)
After dropping zero coords: 39294 rows (-3424)
After dropping lat < 6: 39141 rows (-153)

Final row count: 39141


In [17]:
# ==============================================================
# IMPUTE REMAINING NULLS — per decisions in data_quality_issues.md
# ==============================================================

# Courier age and rating — convert to numeric first, then impute with median
df_clean['courier_age'] = pd.to_numeric(df_clean['courier_age'], errors='coerce')
df_clean['courier_rating'] = pd.to_numeric(df_clean['courier_rating'], errors='coerce')

age_median = df_clean['courier_age'].median()
rating_median = df_clean['courier_rating'].median()

df_clean['courier_age'] = df_clean['courier_age'].fillna(age_median)
df_clean['courier_rating'] = df_clean['courier_rating'].fillna(rating_median)

print(f"Imputed courier_age with median: {age_median}")
print(f"Imputed courier_rating with median: {rating_median}")

# multi_deliveries — null means "no extra deliveries" = 0
df_clean['multi_deliveries'] = pd.to_numeric(df_clean['multi_deliveries'], errors='coerce')
df_clean['multi_deliveries'] = df_clean['multi_deliveries'].fillna(0).astype(int)
print("Imputed multi_deliveries with 0")

# Traffic and weather — impute with mode (most common value)
traffic_mode = df_clean['traffic'].mode()[0]
weather_mode = df_clean['weather'].mode()[0]

df_clean['traffic'] = df_clean['traffic'].fillna(traffic_mode)
df_clean['weather'] = df_clean['weather'].fillna(weather_mode)

print(f"Imputed traffic with mode: {traffic_mode}")
print(f"Imputed weather with mode: {weather_mode}")

# Festival — impute with "No" (base rate)
df_clean['is_festival'] = df_clean['is_festival'].fillna('No')
print("Imputed is_festival with 'No'")

# Final nulls check — should all be 0 now
print("\nRemaining nulls:")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])

Imputed courier_age with median: 30.0
Imputed courier_rating with median: 4.7
Imputed multi_deliveries with 0
Imputed traffic with mode: Low
Imputed weather with mode: Fog
Imputed is_festival with 'No'

Remaining nulls:
Series([], dtype: int64)


In [18]:
# ISSUE 6 FIX- Convert remaining columns to correct types


# delivery_minutes should be integer
df_clean['delivery_minutes'] = pd.to_numeric(df_clean['delivery_minutes'], errors='coerce')

# courier_age should be integer (we already converted to numeric)
df_clean['courier_age'] = df_clean['courier_age'].astype(int)

# order_date should be proper date type
df_clean['order_date'] = pd.to_datetime(df_clean['order_date'], format='%d-%m-%Y', errors='coerce')

# Verify
print(df_clean.dtypes)

order_id                     object
courier_id                   object
courier_age                   int64
courier_rating              float64
restaurant_lat              float64
restaurant_lng              float64
delivery_lat                float64
delivery_lng                float64
order_date           datetime64[ns]
time_ordered                 object
time_picked                  object
weather                      object
traffic                      object
vehicle_condition             int64
order_type                   object
vehicle_type                 object
multi_deliveries              int64
is_festival                  object
city_type                    object
delivery_minutes              int64
dtype: object


In [19]:
# FEATURE ENGINEERING 
# Compute distance (km) using haversine formula
# Explained below - this is the curvature-of-earth distance formula
def haversine_km(lat1, lng1, lat2, lng2):
    R = 6371  # Earth's radius in km
    lat1, lng1, lat2, lng2 = np.radians([lat1, lng1, lat2, lng2])
    dlat = lat2 - lat1
    dlng = lng2 - lng1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlng/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

df_clean['distance_km'] = haversine_km(
    df_clean['restaurant_lat'],
    df_clean['restaurant_lng'],
    df_clean['delivery_lat'],
    df_clean['delivery_lng']
).round(2)

# Extract order_hour from time_ordered (e.g., "11:30:00" -> 11)
df_clean['order_hour'] = pd.to_datetime(
    df_clean['time_ordered'], format='%H:%M:%S', errors='coerce'
).dt.hour

# Day of week (0=Monday, 6=Sunday)
df_clean['day_of_week'] = df_clean['order_date'].dt.day_name()

# Weekend flag
df_clean['is_weekend'] = df_clean['order_date'].dt.dayofweek >= 5

# Peak hour flag - typical lunch (12-14) and dinner (19-21) rushes
df_clean['is_peak_hour'] = df_clean['order_hour'].isin([12, 13, 19, 20, 21])

# On-time flag - SLA is 30 minutes
df_clean['is_on_time'] = (df_clean['delivery_minutes'] <= 30).astype(int)

print("New features added:")
print(df_clean[['distance_km', 'order_hour', 'day_of_week', 'is_weekend', 
                'is_peak_hour', 'is_on_time']].head())

New features added:
   distance_km  order_hour day_of_week  is_weekend  is_peak_hour  is_on_time
0         3.03          11    Saturday        True         False           1
1        20.18          19      Friday       False          True           0
2         1.55           8    Saturday        True         False           1
3         7.79          18     Tuesday       False         False           1
4         6.21          13    Saturday        True          True           1


In [20]:
# OUTLIER HANDLING - cap extreme delivery times
# Some deliveries show unrealistic values - cap at 99th percentile

p99 = df_clean['delivery_minutes'].quantile(0.99)
print(f"99th percentile of delivery time: {p99} minutes")
print(f"Rows above p99: {(df_clean['delivery_minutes'] > p99).sum()}")

# cap (not drop) because these might be legit extreme cases.
# Capping preserves the row while preventing outliers from skewing charts.
df_clean['delivery_minutes'] = df_clean['delivery_minutes'].clip(upper=p99)

print(f"New max delivery time: {df_clean['delivery_minutes'].max()}")

99th percentile of delivery time: 49.0 minutes
Rows above p99: 372
New max delivery time: 49


In [21]:
# SAVE CLEANED DATA

output_path = "../data/processed/orders_cleaned.csv"
df_clean.to_csv(output_path, index=False)

print(f"Saved cleaned dataset to: {output_path}")
print(f"Final shape: {df_clean.shape}")
print(f"\nColumns: {df_clean.columns.tolist()}")
print(f"\nFirst 3 rows:")
print(df_clean.head(3))

Saved cleaned dataset to: ../data/processed/orders_cleaned.csv
Final shape: (39141, 26)

Columns: ['order_id', 'courier_id', 'courier_age', 'courier_rating', 'restaurant_lat', 'restaurant_lng', 'delivery_lat', 'delivery_lng', 'order_date', 'time_ordered', 'time_picked', 'weather', 'traffic', 'vehicle_condition', 'order_type', 'vehicle_type', 'multi_deliveries', 'is_festival', 'city_type', 'delivery_minutes', 'distance_km', 'order_hour', 'day_of_week', 'is_weekend', 'is_peak_hour', 'is_on_time']

First 3 rows:
  order_id      courier_id  courier_age  courier_rating  restaurant_lat  \
0   0x4607  INDORES13DEL02           37             4.9       22.745049   
1   0xb379  BANGRES18DEL02           34             4.5       12.913041   
2   0x5d6d  BANGRES19DEL01           23             4.4       12.914264   

   restaurant_lng  delivery_lat  delivery_lng order_date time_ordered  ...  \
0       75.892471     22.765049     75.912471 2022-03-19     11:30:00  ...   
1       77.683237     13.043